In [1]:
import numpy as np
import polars as pl
import pandas as pd
import tensorflow as tf
from PyEMD import EMD, CEEMDAN, EEMD
from pyeemd import ceemdan, emd, eemd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from working_data import clean_cols, clean_non_minute_rows, alt_label_df as label_df, normalize_by_window, split_df
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from model_builder_trans import combined_loss


2024-10-22 14:23:37.689359: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-22 14:23:37.915883: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-22 14:23:39.047409: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-10-22 14:23:40.976278: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-10-22 14:23:41.249324: 

In [2]:
auc =  tf.keras.metrics.AUC()
auc.reset_state()
prec = tf.keras.metrics.Precision()
prec.reset_state()

In [3]:
NORMALIZING_WINDOW_SIZE = 180
LABELING_WINDOW_SIZE = 20
POSITIVE_SLOPE = 0.3
LABEL_CUR_CANDLE_MULTIPLIER = 0
LABEL_MEAN_MULTIPLIER = 8
BATCH_SIZE = 32
NUM_IMFS = 8
NUM_TOKENS = 88
LOOKBACK_WINDOW = 256
D_MODEL = 32
FF_DIM = 128
NUM_HEADS = 2

In [4]:
source_csv = "data/GBPUSD/minutes.csv"
working_path = "working"

In [5]:
# df = pd.read_csv(source_csv)
# df = clean_non_minute_rows(df)
# df = clean_cols(df)
# break_point = len(df) - len(df)//20
# df = df[break_point:]
# df = normalize_by_window(
#     df, 
#     window_size=NORMALIZING_WINDOW_SIZE, 
#     normalizing_cols=[
#         'open',
#         'high',
#         'low',
#         'close',
#     ])
# print("labeling")
# df = label_df(df, window_size=LABELING_WINDOW_SIZE, mean_multiplier=LABEL_MEAN_MULTIPLIER, cur_candle_multiplier=LABEL_CUR_CANDLE_MULTIPLIER)
# print(df['target'].value_counts())
# split_df(
#     df=df, 
#     dump_path=working_path, 
#     cols=[
#         'close',
#         'close_normalized',
#         'target'
#     ])

In [6]:
def prepare_data(file_path, num_tokens, window_size=1440, batch_size=32, num_imfs=8, smote=False, shuffle=False, col='close'):
    scaler = MinMaxScaler()
    stand_scaler = StandardScaler()
    emd_range = window_size
    # emd = EMD()
    # Load CSV lazily with Polars
    df_lazy = pl.scan_csv(file_path).select([col, 'target'])
    
    # Collect the dataframe and determine total number of rows
    df_collected = df_lazy.collect()
    total_rows = df_collected.shape[0]
    if smote:
        df_collected = df_collected.with_columns(pl.arange(0, total_rows).alias("index"))
        indices_target_1 = df_collected.filter(
            (pl.col("target") == 1) & (pl.col("index") >= emd_range)
        ).select("index").to_series().to_list()

        # Get indices where target is 0 and >= num_tokens
        indices_target_0 = df_collected.filter(
            (pl.col("target") == 0) & (pl.col("index") >= emd_range)
        ).select("index").to_series().to_list()

        df_collected = df_collected.drop('index')
    
    while True:  # Loop to reshuffle and restart at each epoch
        # Create an array of indices to use for shuffling
        indices = list(range(emd_range, total_rows))
        if smote:
            indices_target_1_complete = []
            while len(indices_target_1_complete) < len(indices_target_0):
                indices_target_1_complete += indices_target_1

            indices_target_1_complete = indices_target_1_complete[:len(indices_target_0)]

            indices = indices_target_0 + indices_target_1_complete

        # Shuffle indices if required
        if shuffle:
            np.random.shuffle(indices)

        input_lists = [[] for _ in range(num_imfs)]
        target_list = []

        for idx in indices:
            #create imfs
            signal = np.array(df_collected[col][idx - emd_range + 1:idx + 1])
            signal = scaler.fit_transform(signal.reshape(-1, 1)).flatten()
            # imfs = ceemdan(signal, num_imfs=num_imfs, ensemble_size=100)
            imfs = eemd(signal, num_imfs=num_imfs)
            # scaled_imfs = []
            # for imf in imfs:
            #     # imf_scaled = scaler.fit_transform(imf.reshape(-1, 1)).flatten()
            #     scaled_imfs.append(imf_scaled)

            # imfs = np.array(scaled_imfs)
            # Check the number of IMFs generated
            cur_num_imfs = imfs.shape[0]

            # If fewer than 8 IMFs are generated, pad with flat signals (zeros)
            if cur_num_imfs < num_imfs:
                flat_signal = np.zeros_like(signal)                
                imfs = np.vstack([imfs] + [flat_signal] * (num_imfs - cur_num_imfs))

            # Fetch the previous `num_prev + 1` rows for the input based on the current index
            input_rows = imfs[:, -num_tokens:]
            target_value = df_collected[idx, -1]  # Get 'target' for the target


            for input_idx in range(num_imfs):
                input_lists[input_idx].append(input_rows[input_idx])
            target_list.append(target_value)

            # Yield once we have enough for a batch
            if len(input_lists[0]) == batch_size:
                # Convert lists to NumPy arrays
                input_arrays = [np.array(input_list) for input_list in input_lists]
                target_array = np.array(target_list)

                # Convert NumPy arrays to TensorFlow tensors
                input_tensors = [tf.reshape(tf.convert_to_tensor(input_array, dtype=tf.float32), (batch_size, num_tokens, 1)) for input_array in input_arrays]
                target_tensor = tf.convert_to_tensor(target_array, dtype=tf.int32)

                yield tuple(input_tensors), target_tensor

                # Reset lists for the next batch
                for inputs in input_lists:
                    inputs.clear()
                target_list.clear()
        
        break

In [7]:
def create_dataset_generator(file_path, batch_size, num_tokens, window_size=LOOKBACK_WINDOW, num_imfs=10, shuffle=False, repeat=False, smote=False, col='close'):
    dataset = tf.data.Dataset.from_generator(
        lambda: prepare_data(file_path, window_size=window_size, batch_size=batch_size, num_imfs=num_imfs, num_tokens=num_tokens, shuffle=shuffle, smote=smote, col=col),
        output_signature=(
            tuple([tf.TensorSpec(shape=(None, num_tokens, 1), dtype=tf.float32) for _ in range(num_imfs)]),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    )
    if repeat:
        dataset = dataset.repeat()
    return dataset 


In [8]:
train_dataset = create_dataset_generator('working/train.csv', batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, shuffle=True, num_imfs=NUM_IMFS, col='close', smote=True)
val_dataset = create_dataset_generator('working/val.csv', batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, num_imfs=NUM_IMFS, col='close')
test_dataset = create_dataset_generator('working/test.csv', batch_size=BATCH_SIZE, num_tokens=NUM_TOKENS, num_imfs=NUM_IMFS, col='close')

In [9]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, concatenate, LayerNormalization, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.layers import MultiHeadAttention, Add, Embedding

# Define the number of IMFs
num_imfs = NUM_IMFS
input_length = NUM_TOKENS  # Adjust this according to the length of your IMFs

class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, maxlen, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(maxlen, d_model)

    def positional_encoding(self, maxlen, d_model):
        positions = np.arange(maxlen)[:, np.newaxis]
        angles = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (angles // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

# Create a CNN branch for each IMF

# Transformer block
def transformer_block(inputs, num_heads, ff_dim, dropout_rate=0.1):
    # Multi-head Self-Attention
    attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=inputs.shape[-1])(inputs, inputs)
    attention_output = Dropout(dropout_rate)(attention_output)
    attention_output = Add()([inputs, attention_output])  # Skip connection
    attention_output = LayerNormalization(epsilon=1e-6)(attention_output)
    
    # Feed Forward Network
    ff_output = Dense(ff_dim, activation='relu')(attention_output)
    ff_output = Dense(inputs.shape[-1])(ff_output)
    ff_output = Dropout(dropout_rate)(ff_output)
    ff_output = Add()([attention_output, ff_output])  # Skip connection
    ff_output = LayerNormalization(epsilon=1e-6)(ff_output)
    
    return ff_output

class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)
    

def create_cnn_branch(input_shape):
    input_layer = Input(shape=input_shape)
    x = Conv1D(filters=8, kernel_size=3, activation='relu', padding="same")(input_layer)
    x = MaxPooling1D(pool_size=2, padding="same")(x)
    x = Dropout(0.1)(x)
    x = Conv1D(filters=16, kernel_size=3, activation='relu', padding="same")(x)
    x = MaxPooling1D(pool_size=2, padding="same")(x)
    x = Dropout(0.1)(x)
    x = Dense(D_MODEL)(x)
    x = PositionalEncoding(x.shape[1], d_model=D_MODEL)(x)
    x = TransformerBlock(D_MODEL, NUM_HEADS, 64)(x, training=True)
    x = Dropout(0.1)(x)
    return input_layer, x

# Create input layers and CNN branches for each IMF
inputs = []
cnn_outputs = []

for i in range(num_imfs):
    input_layer, cnn_output = create_cnn_branch((input_length, 1))
    inputs.append(input_layer)
    cnn_outputs.append(cnn_output)

# Concatenate the CNN outputs (tokens)
concatenated_cnn_outputs = concatenate(cnn_outputs, axis=1)
print(concatenated_cnn_outputs.shape)
# reshaped_output = concatenated_cnn_outputs

# Use Lambda layer to reshape the concatenated outputs for the Transformer
# reshaped_output = Lambda(lambda x: tf.reshape(x, (-1, num_imfs, cnn_outputs[0].shape[-1])))(concatenated_cnn_outputs)

# Pass through Transformer
# transformer_output = transformer_block(concatenated_cnn_outputs, num_heads=NUM_HEADS, ff_dim=FF_DIM)

x = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)(concatenated_cnn_outputs, training=True)
x = Dropout(0.1)(x)
# Flatten Transformer output
# flattened_transformer_output = Flatten()(transformer_output)
# flattened_transformer_output = Dropout(0.1)(flattened_transformer_output)

x = Dense(64, activation='relu')(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = Dropout(0.1)(x)
x = Dense(32, activation='relu')(x)
x = Dropout(0.1)(x)

# Output layer for binary classification
outputs = Dense(1, activation='sigmoid')(x)

# Create the model
model = Model(inputs=inputs, outputs=outputs)

# Compile the model
model.compile(
    optimizer='adam', 
    loss=combined_loss, 
    metrics=['accuracy',auc, prec])




(None, 60, 32)


In [10]:
early_stopping = EarlyStopping(monitor='val_precision_1', 
                               patience=5, # Stops if there's no improvement in precision for 5 epochs
                               mode='max', 
                               verbose=1)

model_checkpoint = ModelCheckpoint('best_dpad_model.keras', 
                                   monitor='val_precision_1', 
                                   save_best_only=True, 
                                   mode='max', 
                                   verbose=1)

# Train the model using the train and validation datasets
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,
    callbacks=[early_stopping, model_checkpoint]
)


# Load the best model after training
model.load_weights('best_dpad_model.keras')

Epoch 1/20


I0000 00:00:1729599836.594416  130332 service.cc:145] XLA service 0x7f7d98011a00 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1729599836.594639  130332 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2024-10-22 14:23:57.096950: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-10-22 14:23:58.439615: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1729599860.678291  130538 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_36', 8 bytes spill stores, 8 bytes spill loads

I0000 00:00:1729599860.726902  130551 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_36', 20 bytes spill stores, 20 bytes spill loads

I0

      1/Unknown 66s 66s/step - accuracy: 0.6875 - auc: 0.3136 - loss: 0.8679 - precision_1: 0.6875

I0000 00:00:1729599889.935007  130332 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  18649/Unknown 4962s 263ms/step - accuracy: 0.6390 - auc: 0.6921 - loss: 0.8394 - precision_1: 0.6234

...

18649/18649 ━━━━━━━━━━━━━━━━━━━━ 5491s 291ms/step - accuracy: 0.6390 - auc: 0.6922 - loss: 0.8394 - precision_1: 0.6234 - val_accuracy: 0.6055 - val_auc: 0.7685 - val_loss: 1.0145 - val_precision_1: 0.0961
Epoch 2/20
18649/18649 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.7025 - auc: 0.7699 - loss: 0.7578 - precision_1: 0.6726

2024-10-22 17:18:58.729310: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-10-22 17:18:58.729477: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-10-22 17:18:58.729572: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1090507118135290937
2024-10-22 17:18:58.729580: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 12056812955608733119
2024-10-22 17:18:58.729585: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10969495973193285615
2024-10-22 17:18:58.729633: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 16026414938682604614
2024-10-


Epoch 2: val_precision_1 improved from 0.09614 to 0.09725, saving model to best_dpad_model.keras


2024-10-22 17:28:02.315350: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2024-10-22 17:28:02.315458: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_6]]
2024-10-22 17:28:02.315517: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 1090507118135290937
2024-10-22 17:28:02.315525: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 12056812955608733119
2024-10-22 17:28:02.315532: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 10969495973193285615
2024-10-22 17:28:02.315540: I tensorflow/core/framework/local_rendezvous.cc:422] Local rendezvous recv item cancelled. Key hash: 16026414938682604614
2024-10-

18649/18649 ━━━━━━━━━━━━━━━━━━━━ 5568s 299ms/step - accuracy: 0.7025 - auc: 0.7699 - loss: 0.7578 - precision_1: 0.6726 - val_accuracy: 0.6154 - val_auc: 0.7702 - val_loss: 1.0249 - val_precision_1: 0.0972
Epoch 3/20
  968/18649 ━━━━━━━━━━━━━━━━━━━━ 1:20:09 272ms/step - accuracy: 0.7082 - auc: 0.7775 - loss: 0.7465 - precision_1: 0.6799